In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from model.model_cifar import CIFAR10Classifier, CIFAR100Classifier
import torch.nn.functional as F

In [2]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # Randomly flip the image horizontally
    transforms.RandomCrop(32, padding=4),  # Randomly crop
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))  # Normalize using CIFAR-10 stats
])

batch_size = 64
train_dataset = datasets.CIFAR100(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR100(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Files already downloaded and verified
Files already downloaded and verified


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CIFAR100Classifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [4]:
# 4. Training Loop with Learning Rate Scheduler
def train_model(model, train_loader, criterion, optimizer, device, num_epochs=10):
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)  # Decrease LR by a factor of 0.1 every 5 epochs
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # Calculate accuracy
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
         
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader)
        epoch_accuracy = 100 * correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%, LR: {scheduler.get_last_lr()[0]:.6f}")


In [5]:
train_model(model, train_loader, criterion, optimizer, device, num_epochs=40)

Epoch [1/40], Loss: 3.8040, Accuracy: 10.95%, LR: 0.001000
Epoch [2/40], Loss: 3.0974, Accuracy: 22.63%, LR: 0.001000
Epoch [3/40], Loss: 2.6487, Accuracy: 31.24%, LR: 0.001000
Epoch [4/40], Loss: 2.3289, Accuracy: 38.23%, LR: 0.001000
Epoch [5/40], Loss: 2.0940, Accuracy: 43.12%, LR: 0.001000
Epoch [6/40], Loss: 1.9249, Accuracy: 47.30%, LR: 0.001000
Epoch [7/40], Loss: 1.7954, Accuracy: 50.17%, LR: 0.001000
Epoch [8/40], Loss: 1.6868, Accuracy: 52.98%, LR: 0.001000
Epoch [9/40], Loss: 1.5840, Accuracy: 55.49%, LR: 0.001000
Epoch [10/40], Loss: 1.5121, Accuracy: 57.22%, LR: 0.001000
Epoch [11/40], Loss: 1.4449, Accuracy: 58.93%, LR: 0.001000
Epoch [12/40], Loss: 1.3798, Accuracy: 60.53%, LR: 0.001000
Epoch [13/40], Loss: 1.3165, Accuracy: 62.18%, LR: 0.001000
Epoch [14/40], Loss: 1.2721, Accuracy: 62.99%, LR: 0.001000
Epoch [15/40], Loss: 1.2258, Accuracy: 64.30%, LR: 0.000100
Epoch [16/40], Loss: 1.0063, Accuracy: 70.79%, LR: 0.000100
Epoch [17/40], Loss: 0.9530, Accuracy: 72.38%, LR

In [7]:
# 5. Evaluation
def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

# Evaluate the model
evaluate_model(model, test_loader, device)

# 6. Save the model
torch.save(model.state_dict(), "model/cifar100_model.pth")

Test Accuracy: 65.22%
